# NEXUS-AURORA: Full Training Run

Trains NEXUS-AURORA for 1.5B tokens using Kaggle T4x2 GPUs.

**Prerequisites:**
- Run `01_data_prep.ipynb` to prepare data
- Add `nexus-data` Kaggle Dataset input (contains train.bin, val.bin, tokenizer.model)

**Expected runtime:** ~30-35 hours on 2x T4 (16GB each)

In [ ]:
# !pip install sentencepiece datasets tqdm pyyaml -q

import sys
sys.path.insert(0, '/kaggle/working/nexus-lm')

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')
print(f'GPU count: {torch.cuda.device_count()}')

In [ ]:
# Paths
TRAIN_BIN  = '/kaggle/input/nexus-data/fineweb_edu_train.bin'
VAL_BIN    = '/kaggle/input/nexus-data/fineweb_edu_val.bin'
CKPT_DIR   = '/kaggle/working/checkpoints'
CONFIG     = '/kaggle/working/nexus-lm/config/nexus_aurora_v1.yaml'

In [ ]:
# Build model
import yaml
from model.model import NexusAurora, AuroraConfig

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

model = NexusAurora(AuroraConfig(**cfg['model']))
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

# DataParallel for 2x T4
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
    print(f'Using {torch.cuda.device_count()} GPUs via DataParallel')
model = model.cuda()

In [ ]:
# Setup training
from data.dataloader import MemmapDataset
from training.trainer import Trainer, TrainerConfig

train_ds = MemmapDataset(TRAIN_BIN, seq_len=cfg['model']['max_seq_len'])
val_ds   = MemmapDataset(VAL_BIN,   seq_len=cfg['model']['max_seq_len'])

trainer_cfg = TrainerConfig(
    batch_size=cfg['training']['batch_size'],
    gradient_accumulation=cfg['training']['gradient_accumulation'],
    max_lr=cfg['training']['max_lr'],
    min_lr=cfg['training']['min_lr'],
    warmup_tokens=cfg['training']['warmup_tokens'],
    stable_tokens=cfg['training']['stable_tokens'],
    decay_tokens=cfg['training']['decay_tokens'],
    grad_clip=cfg['training']['grad_clip'],
    weight_decay=cfg['training']['weight_decay'],
    checkpoint_dir=CKPT_DIR,
    checkpoint_every_tokens=100_000_000,
    log_interval=100,
    val_interval=2000,
    max_tokens=1_500_000_000,
    device='cuda',
    dtype='float16',
)

# Use model.module if DataParallel wrapped
raw_model = model.module if hasattr(model, 'module') else model
trainer = Trainer(raw_model, train_ds, val_ds, trainer_cfg)
print('Trainer ready!')

In [ ]:
# Train
trainer.train()

In [ ]:
# Final evaluation
from data.dataloader import get_dataloader
from evaluation.perplexity import compute_bpb

val_loader = get_dataloader(val_ds, batch_size=16, shuffle=False)
device = torch.device('cuda')
bpb = compute_bpb(raw_model, val_loader, device, torch.float16)
print(f'Final BPB: {bpb:.4f}  (target: ≤ 0.94)')